# Building a Multi-Agent DevOps System

By the end of this notebook you'll have a 3-agent pipeline that **reads production logs**, **searches for solutions online**, and **writes a step-by-step remediation plan** — all automatically.

We build it one concept at a time.

| Step | Concept | What It Unlocks |
|------|---------|----------------|
| 1 | **Agent** | A specialized AI worker with a role and goal |
| 2 | **Task** | A specific job you assign to an agent |
| 3 | **Tool** | Gives the agent real-world capabilities (read files, search web) |
| 4 | **Parameters** | Controls for iteration limits, rate limiting, timeouts |
| 5 | **Context** | Output from one task flows into the next |
| 6 | **Crew** | Orchestrates multiple agents working together |

In [1]:
import os

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from dotenv import load_dotenv

load_dotenv()

llm = LLM(
    model="openai/gpt-4o",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

---

## Step 1: Your First Agent

An **Agent** is a specialized AI worker. You define *who* it is (role, backstory) and *what* it's trying to do (goal). The LLM does the rest.

In [2]:
log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    verbose=True,
)

print(f"Agent created: {log_analyzer.role}")

Agent created: DevOps Log Analyzer


---

## Step 2: Your First Task

An agent without a job does nothing. A **Task** tells the agent exactly what to do and what to produce. We'll pass a hardcoded log snippet directly in the task description.

In [3]:
LOG_INPUT = """
[2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment
[2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state
[2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start
[2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'
[2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff
[2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline
[2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints
[2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated
"""

analyze_task = Task(
    description=f"Analyze the following log data to identify issues:\n{LOG_INPUT}",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

crew = Crew(
    agents=[log_analyzer],
    tasks=[analyze_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff()
print(result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the following log data to identify issues:                                                       │
│                                                                                                                 │
│  [2024-01-15 14:32:15.123] INFO: Starting deployment of myapp-deployment                                        │
│  [2024-01-15 14:32:16.567] WARNING: Pod myapp-deployment-7b8c9d5f4-abc12 in Pending state                       │
│  [2024-01-15 14:32:17.890] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  [2024-01-15 14:32:18.123] ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not  │
│  exist or may require 'docker login'                                                                            │
│  [2024-01-15 14:32:18.456] ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  [2024-01-15 14:32:25.901] ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  [2024-01-15 14:32:26.789] WARNING: Service myapp-service has no available endpoints                            │
│  [2024-01-15 14:32:29.456] CRITICAL: Production deployment failed - rollback initiated                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Detailed Analysis Report                                                                                       │
│                                                                                                                 │
│  **Primary Issue Description:**                                                                                 │
│  The deployment of "myapp-deployment" failed primarily due to an inability to pull the required Docker image.   │
│  The failure to pull the image led to the pod being stuck in a pending state and eventually resulting in a      │
│  deployment failure. This ultimately affected the service's availability.                                       │
│                                                                                                                 │
│  **Key Error Messages and Codes:**                                                                              │
│  1. `ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require   │
│  'docker login'`                                                                                                │
│  2. `ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`                                      │
│  3. `ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline`            │
│  4. `WARNING: Service myapp-service has no available endpoints`                                                 │
│  5. `CRITICAL: Production deployment failed - rollback initiated`                                               │
│                                                                                                                 │
│  **Timeline of Failure Events:**                                                                                │
│  - **14:32:15.123:** Deployment process started for "myapp-deployment."                                         │
│  - **14:32:16.567:** Pod "myapp-deployment-7b8c9d5f4-abc12" entered a pending state.                            │
│  - **14:32:17.890:** Pod failed to start.                                                                       │
│  - **14:32:18.123:** Encountered error pulling Docker image "myapp:v1.2.3."                                     │
│  - **14:32:18.456:** Pod status changed to "ImagePullBackOff."                                                  │
│  - **14:32:25.901:** Deployment rollout failed due to exceeding the progress deadline.                          │
│  - **14:32:26.789:** Service "myapp-service" reported no available endpoints.                                   │
│  - **14:32:29.456:** Deployment failure critical, rollback initiated.                                           │
│                                                                                                                 │
│  **Root Cause Analysis:**                                                                                       │
│  The root cause of the deployment failure lies in the inability to access the required Docker image             │
│  "myapp:v1.2.3." The error indicates either the repository does not exist or access is denied due to            │
│  authentication issues, suggesting a failure in the image registry configuration or permissions. This issue     │
│  led to the pod being unable to transition from the pen

Detailed Analysis Report  

**Primary Issue Description:**  
The deployment of "myapp-deployment" failed primarily due to an inability to pull the required Docker image. The failure to pull the image led to the pod being stuck in a pending state and eventually resulting in a deployment failure. This ultimately affected the service's availability.

**Key Error Messages and Codes:**  
1. `ERROR: Failed to pull image "myapp:v1.2.3": pull access denied, repository does not exist or may require 'docker login'`  
2. `ERROR: Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff`  
3. `ERROR: Deployment rollout failed: deployment "myapp-deployment" exceeded its progress deadline`  
4. `WARNING: Service myapp-service has no available endpoints`  
5. `CRITICAL: Production deployment failed - rollback initiated`

**Timeline of Failure Events:**  
- **14:32:15.123:** Deployment process started for "myapp-deployment."
- **14:32:16.567:** Pod "myapp-deployment-7b8c9d5f4-abc12" entered a pend

One agent, one task, one crew — and it produced a full analysis. But we hardcoded the log. What if we want the agent to **read files on its own**?

---

## Step 3: Add a Tool

**Tools** give agents real-world capabilities. `FileReadTool` lets the agent read any file you point it to. We pass the file path as an *input variable* using `{log_file_path}` in the task description.

In [9]:
from crewai_tools import FileReadTool

log_reader_tool = FileReadTool()

In [10]:
log_analyzer_with_tool = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    tools=[log_reader_tool],
    verbose=True,
)

analyze_file_task = Task(
    description="""Analyze the log file at {log_file_path} to identify and extract specific issues.
    
    Your analysis should:
    1. Read through the entire log file carefully
    2. Identify all ERROR, CRITICAL, and WARNING messages
    3. Extract the main issue or failure pattern
    4. Determine the timeline of events leading to the failure
    5. Identify the root cause from the log entries""",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer_with_tool,
)

crew = Crew(
    agents=[log_analyzer_with_tool],
    tasks=[analyze_file_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff(inputs={"log_file_path": "../dummy_logs/kubernetes_deployment_error.log"})
print(result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the log file at ../dummy_logs/kubernetes_deployment_error.log to identify and extract specific   │
│  issues.                                                                                                        │
│                                                                                                                 │
│      Your analysis should:                                                                                      │
│      1. Read through the entire log file carefully                                                              │
│      2. Identify all ERROR, CRITICAL, and WARNING messages                                                      │
│      3. Extract the main issue or failure pattern                                                               │
│      4. Determine the timeline of events leading to the failure                                                 │
│      5. Identify the root cause from the log entries                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Thought: Thought: To analyze the log file appropriately, I need to read the entire contents of the log file    │
│  to identify ERROR, CRITICAL, and WARNING messages, as well as the timeline of events and root cause.           │
│                                                                                                                 │
│  Using Tool: Read a file's content                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"file_path\": \"../dummy_logs/kubernetes_deployment_error.log\", \"start_line\": null, \"line_count\": nul  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  2024-10-10T14:32:15.123Z [INFO] Starting deployment of myapp:v1.2.3                                            │
│  2024-10-10T14:32:15.456Z [INFO] Applying deployment configuration...                                           │
│  2024-10-10T14:32:15.789Z [INFO] Creating deployment myapp-deployment in namespace production                   │
│  2024-10-10T14:32:16.012Z [INFO] Deployment created successfully                                                │
│  2024-10-10T14:32:16.234Z [INFO] Waiting for pods to be ready...                                                │
│  2024-10-10T14:32:16.567Z [WARNING] Pod myapp-deployment-7b8c9d5f4-abc12 is in Pending state                    │
│  2024-10-10T14:32:17.890Z [ERROR] Pod myapp-deployment-7b8c9d5f4-abc12 failed to start                          │
│  2024-10-10T14:32:18.123Z [ERROR] Event: Failed to pull image "myapp:v1.2.3": rpc error: code = Unknown desc =  │
│  Error response from daemon: pull access denied for myapp, repository does not exist or may require 'docker     │
│  login'                                                                                                         │
│  2024-10-10T14:32:18.456Z [ERROR] Pod myapp-deployment-7b8c9d5f4-abc12 status: ImagePullBackOff                 │
│  2024-10-10T14:32:19.789Z [WARNING] Back-off pulling image "myapp:v1.2.3"                                       │
│  2024-10-10T14:32:20.012Z [ERROR] kubelet: Failed to pull image "myapp:v1.2.3": rpc error: code = Unknown desc  │
│  = Error response from daemon: pull access denied for myapp, repository does not exist or may require 'docker   │
│  login'                                                                                                         │
│  2024-10-10T14:32:21.345Z [ERROR] kubelet: Error syncing pod: ErrImagePull                                      │
│  2024-10-10T14:32:22.678Z [WARNING] Pod myapp-deployment-7b8c9d5f4-abc12 has been in ImagePullBackOff state     │
│  for 5 seconds                                                                                                  │
│  2024-10-10T14:32:25.901Z [ERROR] Deployment rollout failed: deployment "myapp-deployment" exceeded its         │
│  progress deadline                                                                                              │
│  2024-10-10T14:32:26.234Z [ERROR] ReplicaSet myapp-deployment-7b8c9d5f4 has 0 ready replicas out of 3 desired   │
│  2024-10-10T14:32:26.567Z [INFO] Current deployment status: 0/3 pods ready                                      │
│  2024-10-10T14:32:27.890Z [WARNING] Deployment health check failed: no healthy pods found                       │
│  2024-10-10T14:32:28.123Z [ERROR] Service myapp-service has no available endpoints                              │
│  2024-10-10T14:32:29.456Z [CRITICAL] Production deployment failed - rollback initiated                          │
│  2024-10-10T14:32:30.789Z [INFO] Rolling back to previous version myapp:v1.2.2                                  │
│  2024-10-1...                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  - **Primary issue description:** The primary issue in the deployment of `myapp:v1.2.3` was an image pull       │
│  failure which resulted in the deployment not having any available ready pods, ultimately leading to a          │
│  deployment failure.                                                                                            │
│                                                                                                                 │
│  - **Key error messages and codes:**                                                                            │
│    - `ERROR` Pod failed to start due to `ImagePullBackOff` state.                                               │
│    - Error message: `Failed to pull image "myapp:v1.2.3": rpc error: code = Unknown desc = Error response from  │
│  daemon: pull access denied for myapp, repository does not exist or may require 'docker login'`.                │
│    - `ERROR` deployment "myapp-deployment" exceeded its progress deadline.                                      │
│    - `ERROR` Service `myapp-service` has no available endpoints.                                                │
│    - `CRITICAL` Production deployment failed - rollback initiated.                                              │
│                                                                                                                 │
│  - **Timeline of failure events:**                                                                              │
│    - [2024-10-10T14:32:15.123Z] Started deployment of `myapp:v1.2.3`.                                           │
│    - [2024-10-10T14:32:16.567Z] Warning of a pod in pending state.                                              │
│    - [2024-10-10T14:32:17.890Z to 2024-10-10T14:32:21.345Z] Multiple errors related to failure to pull the      │
│  image.                                                                                                         │
│    - [2024-10-10T14:32:25.901Z] Deployment rollout failed due to exceeding the progress deadline.               │
│    - [2024-10-10T14:32:28.123Z to 2024-10-10T14:32:29.456Z] Service endpoint failure followed by a critical     │
│  rollback initiated.                                                                                            │
│    - [2024-10-10T14:32:30.789Z] Rollback to previous version completed.                                         │
│                                                                                                                 │
│  - **Root cause analysis:** The root cause of the failure was a denied access to pull the Docker image          │
│  `myapp:v1.2.3`, which resulted from either a missing image in the repository or lacking the necessary          │
│  credentials to access it.                                                                                      │
│                                                                                                                 │
│  - **Affected components:**                                                                                     │
│    - `myapp:v1.2.3` deployment in the `production` namespace.                                                   │
│    - `myapp-deployment` and all associated pods.                                                                │
│    - `myapp-service` due to lack of endpoints from unav

- **Primary issue description:** The primary issue in the deployment of `myapp:v1.2.3` was an image pull failure which resulted in the deployment not having any available ready pods, ultimately leading to a deployment failure.

- **Key error messages and codes:**
  - `ERROR` Pod failed to start due to `ImagePullBackOff` state.
  - Error message: `Failed to pull image "myapp:v1.2.3": rpc error: code = Unknown desc = Error response from daemon: pull access denied for myapp, repository does not exist or may require 'docker login'`.
  - `ERROR` deployment "myapp-deployment" exceeded its progress deadline.
  - `ERROR` Service `myapp-service` has no available endpoints.
  - `CRITICAL` Production deployment failed - rollback initiated.

- **Timeline of failure events:**
  - [2024-10-10T14:32:15.123Z] Started deployment of `myapp:v1.2.3`.
  - [2024-10-10T14:32:16.567Z] Warning of a pod in pending state.
  - [2024-10-10T14:32:17.890Z to 2024-10-10T14:32:21.345Z] Multiple errors related to failu

The agent used `FileReadTool` to read the log file, then analyzed it. With `verbose=True` on the agent, you can see each tool call in the output above.

---

## Step 4: Agent Parameters

Agents can get stuck in loops, burn through API credits, or run forever. These parameters keep them in check.

| Parameter | What It Does |
|-----------|-------------|
| `max_iter` | Max reasoning loops before the agent must produce a final answer |
| `max_rpm` | Rate limit — max LLM requests per minute |
| `max_execution_time` | Hard timeout in seconds |
| `respect_context_window` | Auto-summarize if conversation exceeds the model's context window |

In [11]:
log_analyzer = Agent(
    role="DevOps Log Analyzer",
    goal="Analyze log files to identify and extract specific issues, errors, and failure patterns",
    llm=llm,
    backstory="""You are a senior DevOps engineer with 10 years of experience in 
    analyzing production logs and identifying critical issues. You excel at parsing 
    through complex log files, identifying error patterns, extracting relevant error 
    messages, and determining the root cause of failures from log data.""",
    tools=[log_reader_tool],
    verbose=True,
    max_iter=3,
    max_rpm=10,
    max_execution_time=300,
    respect_context_window=True,
)

print(f"Agent: {log_analyzer.role}")
print(f"  max_iter={log_analyzer.max_iter}, max_rpm={log_analyzer.max_rpm}")
print(f"  max_execution_time={log_analyzer.max_execution_time}s")

Agent: DevOps Log Analyzer
  max_iter=3, max_rpm=10
  max_execution_time=300s


---

## Step 5: Second Agent + Search Tool

One agent can only do so much. Our Log Analyzer identifies the problem, but it can't search the internet for solutions. Let's create a second agent with the **EXA Search Tool**.

In [12]:
from crewai_tools import EXASearchTool

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY")
exa_search_tool = EXASearchTool()

In [13]:
issue_investigator = Agent(
    role="DevOps Issue Investigator",
    goal="Investigate identified issues by searching documentation, forums, and known solutions online",
    llm=llm,
    backstory="""You are a DevOps troubleshooting specialist who excels at quickly 
    finding solutions to technical problems. You know how to search effectively for 
    similar issues, identify reliable sources, and gather comprehensive information 
    about error patterns and their solutions.""",
    tools=[exa_search_tool],
    verbose=True,
    max_iter=5,
    max_rpm=15,
    max_execution_time=600,
    respect_context_window=True,
)

print(f"Agent created: {issue_investigator.role}")

Agent created: DevOps Issue Investigator


---

## Step 6: Context Passing

This is the key to multi-agent systems. The `context` parameter on a Task tells CrewAI: *"Before this agent starts, give it the output from these other tasks."*

The Issue Investigator doesn't need to read the log file itself — it receives the Log Analyzer's analysis automatically.

In [14]:
analyze_task = Task(
    description="""Analyze the log file at {log_file_path} to identify and extract specific issues.
    
    Your analysis should:
    1. Read through the entire log file carefully
    2. Identify all ERROR, CRITICAL, and WARNING messages
    3. Extract the main issue or failure pattern
    4. Determine the timeline of events leading to the failure
    5. Identify the root cause from the log entries""",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

investigate_task = Task(
    description="""Based on the log analysis findings, investigate the identified issue online.
    
    Your investigation should:
    1. Search for similar errors and issues in documentation and forums
    2. Find official documentation related to the error
    3. Look for community solutions and best practices
    4. Identify common causes and scenarios for this type of issue
    5. Gather information about proven fixes and workarounds""",
    expected_output="""A comprehensive investigation report including:
    - Similar issues found online with references
    - Official documentation links
    - Common causes ranked by likelihood
    - Community-verified solutions
    - Best practices to prevent similar issues""",
    agent=issue_investigator,
    context=[analyze_task],
)

crew = Crew(
    agents=[log_analyzer, issue_investigator],
    tasks=[analyze_task, investigate_task],
    process=Process.sequential,
    verbose=False,
)

result = crew.kickoff(inputs={"log_file_path": "../dummy_logs/kubernetes_deployment_error.log"})
print(result.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the log file at ../dummy_logs/kubernetes_deployment_error.log to identify and extract specific   │
│  issues.                                                                                                        │
│                                                                                                                 │
│      Your analysis should:                                                                                      │
│      1. Read through the entire log file carefully                                                              │
│      2. Identify all ERROR, CRITICAL, and WARNING messages                                                      │
│      3. Extract the main issue or failure pattern                                                               │
│      4. Determine the timeline of events leading to the failure                                                 │
│      5. Identify the root cause from the log entries                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 

I encountered an error while trying to use the tool. This was the error: Arguments validation failed: 2 validation errors for FileReadToolSchema
start_line
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='None', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing
line_count
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='None', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing.
 Tool Read a file's content accepts these inputs: Tool Name: Read a file's content
Tool Arguments: {'file_path': {'description': 'Mandatory file full path to read the file', 'type': 'str'}, 'start_line': {'description': 'Line number to start reading from (1-indexed)', 'type': 'Union[int, NoneType]'}, 'line_count': {'description': 'Number of lines to read. If None, reads the entire file', 'type': 'Union[int, No

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Thought: In order to perform the log analysis, I need to start by reading the entire content of the log file   │
│  to understand the context and identify the relevant error patterns.                                            │
│                                                                                                                 │
│  Using Tool: Read a file's content                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"file_path\": \"../dummy_logs/kubernetes_deployment_error.log\", \"start_line\": \"None\", \"line_count\":  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│  I encountered an error while trying to use the tool. This was the error: Arguments validation failed: 2        │
│  validation errors for FileReadToolSchema                                                                       │
│  start_line                                                                                                     │
│    Input should be a valid integer, unable to parse string as an integer [type=int_parsing,                     │
│  input_value='None', input_type=str]                                                                            │
│      For further information visit https://errors.pydantic.dev/2.12/v/int_parsing                               │
│  line_count                                                                                                     │
│    Input should be a valid integer, unable to parse string as an integer [type=int_parsing,                     │
│  input_value='None', input_type=str]                                                                            │
│      For further information visit https://errors.pydantic.dev/2.12/v/int_parsing.                              │
│   Tool Read a file's content accepts these inputs: Tool Name: Read a file's content                             │
│  Tool Arguments: {'file_path': {'description': 'Mandatory file full path to read the file', 'type': 'str'},     │
│  'start_line': {'description': 'Line number to start reading from (1-indexed)', 'type': 'Union[int,             │
│  NoneType]'}, 'line_count': {'description': 'Number of lines to read. If None, reads the entire file', 'type':  │
│  'Union[int, NoneType]'}}                                                                                       │
│  Tool Description: A tool that reads the content of a file. To use this tool, provide a 'file_path' parameter   │
│  with the path to the file you want to read. Optionally, provide 'start_line' to start reading from a specific  │
│  line and 'line_count' to limit the number of lines read..                                                      │
│  Moving on then. I MUST either use a tool (use one at time) OR give my best final answer not both at the same   │
│  time. When responding, I must use the following format:                                                        │
│                                                                                                                 │
│  ```                                                                                                            │
│  Thought: you should always think about what to do                                                              │
│  Action: the action to take, should be one of [Read a file's content]                                           │
│  Action Input: the input to the action, dictionary enclosed in curly braces                                     │
│  Observation: the result of the action                                                                          │
│  ```                                                                                                            │
│  This Thought/Action/Action Input/Result can repeat N times. Once I know the final answer, I must return the    │
│  following format:                                                                                              │
│                                                                                                                 │
│  ```                                                                                                            │
│  Thought: I now can give a great answer               

...

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Issue Investigator                                                                               │
│                                                                                                                 │
│  Thought: Thought: The first step is to search for similar issues online regarding Docker image pulling errors  │
│  in Kubernetes, specifically focusing on the error "pull access denied for myapp, repository does not exist or  │
│  may require 'docker login'".                                                                                   │
│                                                                                                                 │
│  Using Tool: EXASearchTool                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"search_query\": \"Kubernetes pod deployment error pull access denied docker image\", \"start_published_da  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Title: How to Fix the “Pull Access Denied” Error in Kubernetes - Bobcares                                      │
│  URL: https://bobcares.com/blog/kubernetes-pull-access-denied/                                                  │
│  ID: https://bobcares.com/blog/kubernetes-pull-access-denied/                                                   │
│  Score: None                                                                                                    │
│  Published Date: 2025-04-07T00:00:00.000Z                                                                       │
│  Author: None                                                                                                   │
│  Image: None                                                                                                    │
│  Favicon: None                                                                                                  │
│  Extras: None                                                                                                   │
│  Subpages: None                                                                                                 │
│  Text: None                                                                                                     │
│  Highlights: None                                                                                               │
│  Highlight Scores: None                                                                                         │
│  Summary: None                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Title: How to Troubleshoot ImagePullBackOff Errors | K8s Recipes                                               │
│  URL: https://kubernetes.recipes/recipes/troubleshooting/imagepullbackoff-troubleshooting                       │
│  ID: https://kubernetes.recipes/recipes/troubleshooting/imagepullbackoff-troubleshooting                        │
│  Score: None                                                                                                    │
│  Published Date: 2026-01-22T00:00:00.000Z                                                                       │
│  Author: Luca Berton                                                                                            │
│  Image: https://kubernetes.recipes/opengraph.jpg                                                                │
│  Favicon: https://kubernetes.recipes/favicon-32x32.png                                                          │
│  Extras: None                                                                                                   │
│  Subpages: None                                                                                                 │
│  Text: None                                                                                                     │
│  Highlights: None                                                                                               │
│  Highlight Scores: None                                                                                         │
│  Summary: None                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  Title: Getting &quot;ErrImagePull&quot; Error when cr

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Issue Investigator                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Investigation Report on Kubernetes Deployment Error Log**                                      │
│                                                                                                                 │
│  ### Similar Issues Found Online with References                                                                │
│                                                                                                                 │
│  1. **How to Fix the “Pull Access Denied” Error in Kubernetes**: This article provides a step-by-step guide to  │
│  troubleshooting and fixing pull access denied errors in Kubernetes.                                            │
│     [Source](https://bobcares.com/blog/kubernetes-pull-access-denied/)                                          │
│                                                                                                                 │
│  2. **How to Troubleshoot ImagePullBackOff Errors | K8s Recipes**: A detailed guide on understanding and        │
│  resolving ImagePullBackOff errors commonly faced during deployments.                                           │
│     [Source](https://kubernetes.recipes/recipes/troubleshooting/imagepullbackoff-troubleshooting)               │
│                                                                                                                 │
│  3. **Getting "ErrImagePull" Error when creating deployment object (Stack Overflow)**: A community discussion   │
│  about encountering ErrImagePull errors with insights into potential solutions.                                 │
│     [Source](https://stackoverflow.com/questions/78387966/getting-errimagepull-error-when-creating-deployment-  │
│  object)                                                                                                        │
│                                                                                                                 │
│  ### Official Documentation Links                                                                               │
│                                                                                                                 │
│  1. **Troubleshoot image pulls | Google Kubernetes Engine (GKE) | Google Cloud Documentation**: This official   │
│  documentation provides guidance on troubleshooting image pull errors in Kubernetes environments.               │
│     [Source](https://cloud.google.com/kubernetes-engine/docs/troubleshooting/image-pulls)                       │
│                                                                                                                 │
│  ### Common Causes Ranked by Likelihood                                                                         │
│                                                                                                                 │
│  1. **Incorrect Docker Repository Credentials**: Misconfigured or missing credentials for accessing the Docker  │
│  repository can lead to access denial.                                                                          │
│  2. **Non-Existent Repository**: The specified Docker repository or image might not exist.                      │
│  3. **Docker Image Tag Mismatch**: Specifying incorrect tags for images during Kubernetes deployment.           │
│  4. **Network Connectivity Issues**: Intermittent netwo

**Comprehensive Investigation Report on Kubernetes Deployment Error Log**

### Similar Issues Found Online with References

1. **How to Fix the “Pull Access Denied” Error in Kubernetes**: This article provides a step-by-step guide to troubleshooting and fixing pull access denied errors in Kubernetes.
   [Source](https://bobcares.com/blog/kubernetes-pull-access-denied/)

2. **How to Troubleshoot ImagePullBackOff Errors | K8s Recipes**: A detailed guide on understanding and resolving ImagePullBackOff errors commonly faced during deployments.
   [Source](https://kubernetes.recipes/recipes/troubleshooting/imagepullbackoff-troubleshooting)

3. **Getting "ErrImagePull" Error when creating deployment object (Stack Overflow)**: A community discussion about encountering ErrImagePull errors with insights into potential solutions.
   [Source](https://stackoverflow.com/questions/78387966/getting-errimagepull-error-when-creating-deployment-object)

### Official Documentation Links

1. **Troubleshoo

Two agents working together. The Investigator's output references the Analyzer's findings — that's context passing at work.

---

## Step 7: Third Agent — The Synthesizer

The final agent takes the analysis *and* the investigation and produces a concrete remediation plan. Notice `context` can reference **multiple** upstream tasks.

In [15]:
solution_specialist = Agent(
    role="DevOps Solution Specialist",
    goal="Provide clear, actionable solutions with step-by-step instructions based on investigation findings",
    llm=llm,
    backstory="""You are a DevOps solutions architect who specializes in creating 
    reliable, step-by-step remediation plans for infrastructure and deployment issues. 
    You always provide official documentation references, tested solutions, and 
    preventive measures to avoid future occurrences.""",
    verbose=True,
    max_iter=4,
    max_rpm=8,
    max_execution_time=450,
    respect_context_window=True,
)

print(f"Agent created: {solution_specialist.role}")
print("This agent has no tools — it synthesizes findings from the other two agents.")

Agent created: DevOps Solution Specialist
This agent has no tools — it synthesizes findings from the other two agents.


---

## Step 8: The Full Pipeline

All three agents. All three tasks. One crew. `Process.sequential` ensures they run in order. `cache=True` avoids redundant LLM calls. `max_rpm=30` is the crew-level rate limit.

In [16]:
analyze_task = Task(
    description="""Analyze the log file at {log_file_path} to identify and extract specific issues.
    
    Your analysis should:
    1. Read through the entire log file carefully
    2. Identify all ERROR, CRITICAL, and WARNING messages
    3. Extract the main issue or failure pattern
    4. Determine the timeline of events leading to the failure
    5. Identify the root cause from the log entries""",
    expected_output="""A detailed analysis report containing:
    - Primary issue description
    - Key error messages and codes
    - Timeline of failure events
    - Root cause analysis
    - Affected components""",
    agent=log_analyzer,
)

investigate_task = Task(
    description="""Based on the log analysis findings, investigate the identified issue online.
    
    Your investigation should:
    1. Search for similar errors and issues in documentation and forums
    2. Find official documentation related to the error
    3. Look for community solutions and best practices
    4. Identify common causes and scenarios for this type of issue
    5. Gather information about proven fixes and workarounds""",
    expected_output="""A comprehensive investigation report including:
    - Similar issues found online with references
    - Official documentation links
    - Common causes ranked by likelihood
    - Community-verified solutions
    - Best practices to prevent similar issues""",
    agent=issue_investigator,
    context=[analyze_task],
)

solution_task = Task(
    description="""Based on the log analysis and investigation findings, provide a complete solution.
    
    Your solution should:
    1. Create a step-by-step remediation plan with specific commands
    2. Provide verification steps to confirm the fix
    3. Suggest monitoring and prevention measures
    4. Include rollback procedures if needed
    5. Reference official documentation""",
    expected_output="""A detailed remediation plan with:
    - Primary solution with step-by-step commands
    - Verification and testing procedures
    - Prevention strategies and monitoring recommendations
    - Rollback plan in case of issues
    - Links to official documentation""",
    agent=solution_specialist,
    context=[analyze_task, investigate_task],
)

devops_crew = Crew(
    agents=[log_analyzer, issue_investigator, solution_specialist],
    tasks=[analyze_task, investigate_task, solution_task],
    process=Process.sequential,
    verbose=True,
    cache=True,
    max_rpm=30,
)

result = devops_crew.kickoff(
    inputs={"log_file_path": "../dummy_logs/kubernetes_deployment_error.log"}
)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 40d2dba4-3031-4915-96ef-826a4c21e24b                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Task: Analyze the log file at ../dummy_logs/kubernetes_deployment_error.log to identify and extract specific   │
│  issues.                                                                                                        │
│                                                                                                                 │
│      Your analysis should:                                                                                      │
│      1. Read through the entire log file carefully                                                              │
│      2. Identify all ERROR, CRITICAL, and WARNING messages                                                      │
│      3. Extract the main issue or failure pattern                                                               │
│      4. Determine the timeline of events leading to the failure                                                 │
│      5. Identify the root cause from the log entries                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Log Analyzer                                                                                     │
│                                                                                                                 │
│  Thought: Thought: To analyze the log file, I first need to read its entire content to identify and extract     │
│  specific issues such as ERROR, CRITICAL, and WARNING messages. This will help in determining the timeline of   │
│  events and identifying the root cause.                                                                         │
│                                                                                                                 │
│  Using Tool: Read a file's content                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  "{\"file_path\": \"../dummy_logs/kubernetes_deployment_error.log\", \"start_line\": null, \"line_count\": nul  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Solution Specialist                                                                              │
│                                                                                                                 │
│  Task: Based on the log analysis and investigation findings, provide a complete solution.                       │
│                                                                                                                 │
│      Your solution should:                                                                                      │
│      1. Create a step-by-step remediation plan with specific commands                                           │
│      2. Provide verification steps to confirm the fix                                                           │
│      3. Suggest monitoring and prevention measures                                                              │
│      4. Include rollback procedures if needed                                                                   │
│      5. Reference official documentation                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DevOps Solution Specialist                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### Complete Solution for Image Pull Error in Kubernetes Deployment                                            │
│                                                                                                                 │
│  #### Step-by-Step Remediation Plan                                                                             │
│                                                                                                                 │
│  1. **Create or Update Docker Registry Credentials:**                                                           │
│     - **Log in to the Docker registry:**                                                                        │
│       ```bash                                                                                                   │
│       docker login <registry_url>                                                                               │
│       ```                                                                                                       │
│     - **Create a Kubernetes Secret using Docker credentials:**                                                  │
│       ```bash                                                                                                   │
│       kubectl create secret docker-registry myapp-registry-secret \                                             │
│         --docker-server=<registry_url> \                                                                        │
│         --docker-username=<your_username> \                                                                     │
│         --docker-password=<your_password> \                                                                     │
│         --docker-email=<your_email>                                                                             │
│       ```                                                                                                       │
│                                                                                                                 │
│  2. **Attach Secret to the Deployment:**                                                                        │
│     - **Edit the Kubernetes deployment YAML to include the image pull secret:**                                 │
│       ```yaml                                                                                                   │
│       spec:                                                                                                     │
│         containers:                                                                                             │
│         - name: myapp                                                                                           │
│           image: myapp:v1.2.3                                                                                   │
│         imagePullSecrets:                                                                                       │
│         - name: myapp-registry-secret                                                                           │
│       ```                                                                                                       │
│                                                                                                                 │
│  3. **Apply the Adjusted Deployment:**                 

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 32b46467-8031-41f8-82f2-8a0e277efbab                                                                     │
│  Agent: DevOps Solution Specialist                                                                              │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 40d2dba4-3031-4915-96ef-826a4c21e24b                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ### Complete Solution for Image Pull Error in Kubernetes Deployment                              │
│                                                                                                                 │
│  #### Step-by-Step Remediation Plan                                                                             │
│                                                                                                                 │
│  1. **Create or Update Docker Registry Credentials:**                                                           │
│     - **Log in to the Docker registry:**                                                                        │
│       ```bash                                                                                                   │
│       docker login <registry_url>                                                                               │
│       ```                                                                                                       │
│     - **Create a Kubernetes Secret using Docker credentials:**                                                  │
│       ```bash                                                                                                   │
│       kubectl create secret docker-registry myapp-registry-secret \                                             │
│         --docker-server=<registry_url> \                                                                        │
│         --docker-username=<your_username> \                                                                     │
│         --docker-password=<your_password> \                                                                     │
│         --docker-email=<your_email>                                                                             │
│       ```                                                                                                       │
│                                                                                                                 │
│  2. **Attach Secret to the Deployment:**                                                                        │
│     - **Edit the Kubernetes deployment YAML to include the image pull secret:**                                 │
│       ```yaml                                                                                                   │
│       spec:                                                                                                     │
│         containers:                                                                                             │
│         - name: myapp                                                                                           │
│           image: myapp:v1.2.3                                                                                   │
│         imagePullSecrets:                                                                                       │
│         - name: myapp-registry-secret                                                                           │
│       ```                                                                                                       │
│                                                       

In [17]:
print(result.raw)

### Complete Solution for Image Pull Error in Kubernetes Deployment

#### Step-by-Step Remediation Plan

1. **Create or Update Docker Registry Credentials:**
   - **Log in to the Docker registry:**
     ```bash
     docker login <registry_url>
     ```
   - **Create a Kubernetes Secret using Docker credentials:**
     ```bash
     kubectl create secret docker-registry myapp-registry-secret \
       --docker-server=<registry_url> \
       --docker-username=<your_username> \
       --docker-password=<your_password> \
       --docker-email=<your_email>
     ```

2. **Attach Secret to the Deployment:**
   - **Edit the Kubernetes deployment YAML to include the image pull secret:**
     ```yaml
     spec:
       containers:
       - name: myapp
         image: myapp:v1.2.3
       imagePullSecrets:
       - name: myapp-registry-secret
     ```

3. **Apply the Adjusted Deployment:**
   - **Update the deployment with the new YAML configuration:**
     ```bash
     kubectl apply -f myapp-deploym

---

## Step 9: Saving Outputs to Files

Add `output_file` to any task and CrewAI writes the result to disk automatically. Useful for creating audit trails or feeding outputs into other systems.

In [18]:
os.makedirs("task_outputs", exist_ok=True)

analyze_task_with_file = Task(
    description="Analyze the log file at {log_file_path} to identify issues.",
    expected_output="A detailed analysis report with root cause and timeline.",
    agent=log_analyzer,
    output_file="task_outputs/log_analysis.md",
)

investigate_task_with_file = Task(
    description="Investigate the identified issue online.",
    expected_output="Investigation report with solutions and references.",
    agent=issue_investigator,
    context=[analyze_task_with_file],
    output_file="task_outputs/investigation_report.md",
)

solution_task_with_file = Task(
    description="Provide a complete solution with commands and verification steps.",
    expected_output="Remediation plan with commands, verification, and rollback.",
    agent=solution_specialist,
    context=[analyze_task_with_file, investigate_task_with_file],
    output_file="task_outputs/solution_plan.md",
)

print("Tasks configured with output_file:")
print(f"  Task 1 → {analyze_task_with_file.output_file}")
print(f"  Task 2 → {investigate_task_with_file.output_file}")
print(f"  Task 3 → {solution_task_with_file.output_file}")
print("\nRun the crew to generate files (same as Step 8, but results are saved to disk).")

Tasks configured with output_file:
  Task 1 → task_outputs/log_analysis.md
  Task 2 → task_outputs/investigation_report.md
  Task 3 → task_outputs/solution_plan.md

Run the crew to generate files (same as Step 8, but results are saved to disk).


---

## Recap

| Step | You Learned | CrewAI API |
|------|------------|------------|
| 1 | Create a specialized agent | `Agent(role, goal, backstory, llm)` |
| 2 | Give it a job | `Task(description, expected_output, agent)` |
| 3 | Give it tools | `tools=[FileReadTool()]`, `{variable}` inputs |
| 4 | Control its behavior | `max_iter`, `max_rpm`, `max_execution_time` |
| 5 | Add more agents | `EXASearchTool()`, multiple agents |
| 6 | Chain tasks together | `context=[upstream_task]` |
| 7 | Synthesize across tasks | `context=[task_1, task_2]` |
| 8 | Run the full pipeline | `Crew(agents, tasks, process, cache, max_rpm)` |
| 9 | Save results to disk | `output_file="path.md"` |

**Next →** Open `02_production_features.ipynb` to add structured output, guardrails, and memory to this pipeline.